# 🔄 Grunnleggende agentarbeidsflyter med Microsoft Foundry (Python)

## 📋 Veiledning for arbeidsflytorchestrering

Denne notatboken introduserer de kraftige **Workflow Builder**-funksjonene i Microsoft Agent Framework. Lær hvordan du kan lage sofistikerte, trinnvise agentarbeidsflyter som kan håndtere komplekse forretningsprosesser og koordinere flere AI-operasjoner sømløst.

> **Migrasjonsmerknad:** Dette eksemplet refererte tidligere til GitHub Models. GitHub Models er avviklet (avsluttes juli 2026), så det bruker nå **Microsoft Foundry** gjennom `FoundryChatClient`, som retter seg mot Azure OpenAI **Responses API**.

## 🎯 Læringsmål

### 🏗️ **Arbeidsflytarkitektur**
- **Workflow Builder**: Design og orkestrer komplekse trinnvise prosesser
- **Hendelsesdrevet utførelse**: Håndter arbeidsflythendelser og tilstandsforandringer
- **Visuell arbeidsflytdesign**: Lag og visualiser arbeidsflytstrukturer
- **Microsoft Foundry-integrasjon**: Utnytt AI-modeller i arbeidsflytkontekster

### 🔄 **Prosesorchestrering**
- **Sekvensielle operasjoner**: Koble sammen flere agentoppgaver i logisk rekkefølge
- **Betinget logikk**: Implementer beslutningspunkter og forgrenede arbeidsflyter
- **Feilhåndtering**: Robust feilgjenoppretting og arbeidsflytresiliens
- **Tilstandsstyring**: Spor og håndter arbeidsflytutførelsens tilstand

### 📊 **Enterprise arbeidsflytmønstre**
- **Automatisering av forretningsprosesser**: Automatiser komplekse organisatoriske arbeidsflyter
- **Koordinering av flere agenter**: Koordiner flere spesialiserte agenter
- **Skalerbar utførelse**: Design arbeidsflyter for bedriftsomfattende operasjoner
- **Overvåking og observabilitet**: Spor arbeidsflytytelse og resultater

## ⚙️ Forutsetninger og oppsett

### 📦 **Nødvendige avhengigheter**

Installer Agent Framework med arbeidsflytfunksjoner:

```bash
pip install agent-framework -U
```

### 🔑 **Microsoft Foundry-konfigurasjon**

Logg inn med Azure CLI (`az login`) slik at `AzureCliCredential` kan autentisere, og sett deretter dine Microsoft Foundry-prosjektdetaljer.

**Miljøoppsett (.env-fil):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4.1-mini
```

### 🏢 **Enterprise brukstilfeller**

**Eksempler på forretningsprosesser:**
- **Kundeinnføring**: Flert-trinn verifikasjon og oppsettsarbeidsflyter
- **Innholdspipeline**: Automatisert innholdsproduksjon, gjennomgang og publisering
- **Databehandling**: ETL-arbeidsflyter med AI-drevet transformasjon
- **Kvalitetssikring**: Automatiserte test- og valideringsprosesser

**Fordeler med arbeidsflyter:**
- 🎯 **Pålitelige**: Deterministisk utførelse med feilgjenoppretting
- 📈 **Skalerbarhet**: Håndter høyt volum av prosessautomatisering
- 🔍 **Observabilitet**: Fullstendige revisjonsspor og overvåking
- 🔧 **Vedlikeholdbarhet**: Visuell design og modulære komponenter

## 🎨 Arbeidsflytdesignmønstre

### Grunnleggende arbeidsflytstruktur
```mermaid
graph TD
    A[Start] --> B[Agentoppgave 1]
    B --> C{Beslutningspunkt}
    C -->|Suksess| D[Agentoppgave 2]
    C -->|Feil| E[Feilhåndtering]
    D --> F[Slutt]
    E --> F
```

**Nøkkelkomponenter:**
- **WorkflowBuilder**: Hovedorkestreringsmotor
- **WorkflowEvent**: Hendelseshåndtering og kommunikasjon
- **WorkflowViz**: Visuell arbeidsflytrepresentasjon og feilsøking

La oss bygge din første intelligente arbeidsflyt! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load Microsoft Foundry project settings from .env file
load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = provider.as_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = provider.as_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
